In [1]:
import pandas as pd

In [2]:
import socket
socket.getaddrinfo('localhost',1521)

[(<AddressFamily.AF_INET6: 23>, 0, 0, '', ('::1', 1521, 0, 0)),
 (<AddressFamily.AF_INET: 2>, 0, 0, '', ('127.0.0.1', 1521))]

In [3]:
from dotenv import load_dotenv
import os
import oracledb
load_dotenv()
ORACLE_NAME=os.getenv('ORACLE_NAME')
ORACLE_PASSWORD=os.getenv('ORACLE_PASSWORD')
ORACLE_HOST=os.getenv('ORACLE_HOST')
ORACLE_PORT=int(os.getenv('ORACLE_PORT'))
ORACLE_SERVICE_NAME=os.getenv('ORACLE_SERVICE_NAME')

In [9]:
conn=oracledb.connect(
    user=ORACLE_NAME,
    password=ORACLE_PASSWORD,
    host=ORACLE_HOST,
    port=ORACLE_PORT,
    service_name=ORACLE_SERVICE_NAME
)
cursor=conn.cursor()

CUSTOMERS INSERTS

In [ ]:
import random
import string
from datetime import date, timedelta

random.seed(42)

adlar = ["Anar", "Elvin", "Tural", "Nicat", "Orxan", "Kamran", "Murad", "Rauf", "Elnur", "Samir",
"Aysel", "Gunel", "Leyla", "Nermin", "Sevinc", "Ayten", "Konul", "Zehra", "Gulnar", "Turkan",
"Ferid", "Vusal", "Resad", "Eli", "Cavid", "Behruz", "Rehin", "Hesen", "Ilkin", "Rovsen",
"Narmin", "Afet", "Lale", "Mele", "Sabina", "Nilufer", "Arzu", "Seda", "Fidan", "Aytac"]
soyadlar = ["Hesenov", "Eliyev", "Huseynov", "Quliyev", "Memmedov", "Ismayilov", "Nasirov", "Babayev",
"Hesenova", "Eliyeva", "Huseynova", "Quliyeva", "Memmedova", "Ismayilova", "Nasirova", "Babayeva",
"Kerimov", "Kerimova", "Rzayev", "Rzayeva", "Haciyev", "Haciyeva",
"Mammadli", "Ibrahimli", "Hasanli", "Huseynli", "Quluzade", "Memmedli"]
ata_adlari = ["Elvin oglu", "Murad oglu", "Tural oglu", "Samir oglu", "Orxan oglu",
"Nicat oglu", "Kamran oglu", "Rauf oglu", "Cavid oglu", "Elnur oglu",
"Elvin qizi", "Murad qizi", "Tural qizi", "Samir qizi", "Orxan qizi",
"Nicat qizi", "Kamran qizi", "Rauf qizi", "Cavid qizi", "Elnur qizi"]
email_domainler = ["gmail.com", "mail.ru", "yahoo.com", "outlook.com", "box.az"]
VALID_PREFIXES  = ["010","050", "051", "055", "060", "070", "077", "099"]

def valid_phone():
    prefix=random.choice(VALID_PREFIXES)
    mid=random.randint(2, 9)
    rest=''.join(random.choices(string.digits, k=6))
    fmt=random.randint(0, 3)
    core=str(mid) + rest
    if fmt==0:return f"+994{prefix[1:]}{core}"
    if fmt==1:return f"994{prefix[1:]}{core}"
    if fmt==2:return f"{prefix}{core}"
    return f"{prefix[1:]}{core}"

def valid_email(ad):
    return f"{ad.lower()}{random.randint(1,999)}@{random.choice(email_domainler)}"

def rand_date(y1,y2):
    start=date(y1,1,1)
    end=date(y2,12,31)
    return start+timedelta(days=random.randint(0,(end-start).days))

total=5000
valid_count=4800
invalid_count=200

serials_set=set()
valid_serials=[]
while len(valid_serials)<valid_count:
    prefix=random.choice(["AA","AB","AZE"])
    digits=''.join(random.choices(string.digits,k=7 if prefix!="AZE" else 8))
    s=prefix+digits
    if s not in serials_set:
        serials_set.add(s)
        valid_serials.append(s)

invalid_templates = [
    lambda:"AAA" +''.join(random.choices(string.digits, k=7)),
    lambda:"ABB" +''.join(random.choices(string.digits, k=7)),
    lambda:"AZEE" +''.join(random.choices(string.digits, k=8)),
    lambda:''.join(random.choices(string.digits, k=9)),
    lambda:"AA" +''.join(random.choices(string.digits, k=5)),
    lambda:"AZE"+''.join(random.choices(string.digits, k=6)),
]
invalid_serials = []
while len(invalid_serials)<invalid_count:
    s=random.choice(invalid_templates)()
    if s not in serials_set:
        serials_set.add(s)
        invalid_serials.append(s)

err_serial_idx=set(random.sample(range(total),invalid_count))

valid_iter=iter(valid_serials)
invalid_iter=iter(invalid_serials)

remaining=[i for i in range(total) if i not in err_serial_idx]
random.shuffle(remaining)

err_swap_name=set(remaining[0:200])
err_dob_future=set(remaining[200:350])
err_score=set(remaining[350:500])
err_null=set(remaining[500:650])
err_phone=set(remaining[650:800])
data=[]

for i in range(total):
    cid=i+1
    ad=random.choice(adlar)
    soyad=random.choice(soyadlar)
    ata=random.choice(ata_adlari)
    email=valid_email(ad)
    phone=valid_phone()
    dob=rand_date(1970, 2000)
    score=random.randint(1, 100)
    created=rand_date(2018, 2024)

    if i in err_swap_name:
        ad,soyad=soyad,ad

    serial=next(invalid_iter) if i in err_serial_idx else next(valid_iter)

    if i in err_dob_future:
        dob=rand_date(2026,2030)

    if i in err_score:
        score=random.choice([random.randint(-100,0),random.randint(101,300)])

    if i in err_null:
        f=random.choice(["email","phone","score","middle"])
        if f=="email":email=None
        if f=="phone":phone=None
        if f=="score":score=None
        if f=="middle":ata=None

    if i in err_phone:
        phone=None

    data.append((cid,ad,soyad,ata,serial,dob,phone,email,score,created))

insert_query = """
INSERT INTO CUSTOMERS
(customer_id, first_name, last_name, middle_name, serial_number,
date_of_birth, phone_number, email, internal_score, created_date)
VALUES (:1,:2,:3,:4,:5,:6,:7,:8,:9,:10)
"""

cursor.executemany(insert_query,data)
conn.commit()

LOAN ACCOUNT INSERTS

In [14]:
LOAN_TYPES_VALID=["PERSONAL","AUTO","MORTGAGE","BUSINESS"]
LOAN_TYPES_INVALID=["STUDENT","MICRO","OVERDRAFT"]
LOAN_STATUS_VALID=["ACTIVE","CLOSED","DEFAULT","DELINQUENT"]
LOAN_STATUS_INVALID=["PENDING","APPROVED","REJECTED","REVIEW"]
TERM_MONTHS=[12,24,36,48,60]
def rand_date(y1,y2):
    start=date(y1,1,1)
    end=date(y2,12,31)
    return start+timedelta(days=random.randint(0,(end-start).days))
def add_months(d,months):
    month=d.month-1+months
    year=d.year+month//12
    month=month%12+1
    return date(year,month,1)
total=5000
err_pool=random.sample(range(total),1000)
random.shuffle(err_pool)
err_loan_type=set(err_pool[0:150])
err_rate=set(err_pool[150:300])
err_end_date=set(err_pool[300:450])
err_status=set(err_pool[450:600])
err_balance=set(err_pool[600:750])
err_created=set(err_pool[750:850])
err_null=set(err_pool[850:1000])
data=[]
for i in range(total):
    loan_id=i+1
    customer_id=random.randint(1,5000)
    term=random.choice(TERM_MONTHS)
    start=rand_date(2019,2024)
    end=add_months(start,term)
    principal=round(random.uniform(500,50000),2)
    rate=round(random.uniform(0.08,0.36),4)
    status=random.choice(LOAN_STATUS_VALID)
    balance=round(principal*random.uniform(0,1),2)
    loan_type=random.choice(LOAN_TYPES_VALID)
    created=rand_date(2019,2024)
    if i in err_loan_type:
        loan_type=random.choice(LOAN_TYPES_INVALID)
    if i in err_rate:
        rate=round(random.uniform(1.01,3.0),4)
    if i in err_end_date:
        end=start-timedelta(days=random.randint(1,365))
    if i in err_status:
        status=random.choice(LOAN_STATUS_INVALID)
    if i in err_balance:
        balance=round(random.uniform(-5000,-1),2)
    if i in err_created:
        created=rand_date(2026,2027)
        while created<=date(2026,4,1):
            created=rand_date(2026,2027)
    if i in err_null:
        f=random.choice(["rate","status","balance","loan_type"])
        if f=="rate":rate=None
        if f=="status":status=None
        if f=="balance":balance=None
        if f=="loan_type":loan_type=None
    data.append((loan_id,customer_id,loan_type,principal,rate,term,start,end,status,balance,created))
insert_query="""
INSERT INTO LOAN_ACCOUNT
(loan_id,customer_id,loan_type,principal_amount,interest_rate,
term_months,start_date,end_date,loan_status,outstanding_balance,created_date)
VALUES (:1,:2,:3,:4,:5,:6,:7,:8,:9,:10,:11)
"""
cursor.executemany(insert_query,data)
conn.commit()

LOAN PAYMENT INSERT

In [ ]:
PAY_STATUS_VALID=["PAID","PARTIAL","MISSED","PENDING"]
PAY_STATUS_INVALID=["APPROVED","FAILED","CANCELLED","PROCESSING"]
PAY_METHOD_VALID=["BANK_TRANSFER","CASH","CARD","AUTO_DEBIT"]
PAY_METHOD_INVALID=["CRYPTO","CHEQUE","BARTER","UNKNOWN"]
def rand_date(y1,y2):
    start=date(y1,1,1)
    end=date(y2,12,31)
    return start+timedelta(days=random.randint(0,(end-start).days))
total=5000
err_pool=random.sample(range(total),1000)
random.shuffle(err_pool)
err_status=set(err_pool[0:250])
err_method=set(err_pool[250:500])
err_amount=set(err_pool[500:750])
err_created=set(err_pool[750:900])
err_null=set(err_pool[900:1000])
data=[]
for i in range(total):
    payment_id=i+1
    loan_id=random.randint(1,5000)
    due=rand_date(2019,2025)
    pay_date=due+timedelta(days=random.randint(0,10))
    amount_due=round(random.uniform(100,2000),2)
    amount_paid=round(amount_due*random.uniform(0,1),2)
    method=random.choice(PAY_METHOD_VALID)
    status=random.choice(PAY_STATUS_VALID)
    created=rand_date(2019,2025)
    if i in err_status:
        status=random.choice(PAY_STATUS_INVALID)
    if i in err_method:
        method=random.choice(PAY_METHOD_INVALID)
    if i in err_amount:
        amount_paid=round(random.uniform(-2000,-1),2)
    if i in err_created:
        created=rand_date(2026,2027)
        while created<=date(2026,4,1):
            created=rand_date(2026,2027)
    if i in err_null:
        f=random.choice(["pay_date","method","status","amount_paid"])
        if f=="pay_date":pay_date=None
        if f=="method":method=None
        if f=="status":status=None
        if f=="amount_paid":amount_paid=None
    data.append((payment_id,loan_id,due,pay_date,amount_due,amount_paid,method,status,created))
insert_query="""
INSERT INTO LOAN_PAYMENT
(payment_id,loan_id,due_date,payment_date,amount_due,amount_paid,
payment_method,payment_status,created_date)
VALUES (:1,:2,:3,:4,:5,:6,:7,:8,:9)
"""
cursor.executemany(insert_query,data)
conn.commit()

: 